<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">Lab 02: 最初の Prompt Agent を作成する</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
  Microsoft Foundry Agent Observability
  </p>
</div>

## 学習内容

このラボでは、Azure AI Projects SDK を使用して最初の **Prompt Agent** — **Contoso Travel コンシェルジュ** — を作成します。カスタム指示を持つエージェントの定義方法、会話の開始、メッセージの送信、マルチターン対話の処理方法を学びます。コンシェルジュは顧客を迎え、一般的な旅行の質問に答えるフロントデスクエージェントです。後続のラボでは、専門ツールを追加し、ドメイン固有のエージェントと連携させます。

---

このラボでは、ゼロから動作する **Prompt Agent** — Contoso Travel コンシェルジュ — を構築します:
 - `PromptAgentDefinition` でエージェントを定義する、
 - 会話を作成してメッセージを送信する、
 - マルチターン対話を処理する、
 - レスポンスオブジェクトを検査する。

## 1. セットアップ

環境変数を読み込み、Azure AI Project クライアントを作成します。

---


In [ ]:
# セットアップ: 認証情報を読み込み SDK クライアントを作成
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, AzureCliCredential
from azure.ai.projects import AIProjectClient
# PromptAgentDefinition: LLM エージェントの宣言的設定
# （モデル + 指示 + オプションツール）
from azure.ai.projects.models import PromptAgentDefinition

# .env はノートブックディレクトリの一つ上に存在
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

tenant_id = os.getenv("TENANT_ID")
endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = AzureCliCredential(tenant_id=tenant_id)
#credential = DefaultAzureCredential()
# AIProjectClient はエージェントを管理し、openai_client は
# OpenAI 互換 API 経由で会話とレスポンスを処理する
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Microsoft Foundry に接続しました")
print(f"   モデル: {model_name}")

## 2. コンシェルジュエージェントの作成

旅行に特化した指示でエージェントを定義します。システムプロンプトは顧客クエリへの応答方法を形成します。

---


In [ ]:
# システムプロンプトを定義 — エージェントのパーソナリティ・
# 制約・応答スタイルを形成する
CONCIERGE_INSTRUCTIONS = """あなたは Contoso Travel のトラベルコンシェルジュです。親しみやすく、知識豊富な旅行アシスタントとして対応してください。

あなたの役割：
- 旅行先、旅行のコツ、移動手段などについての質問に答え、お客様の旅行計画をサポートする
- 役立ち、正確で簡潔な旅行アドバイスを提供する
- 温かみがありつつ、プロフェッショナルな対応を心がける
- 特定の情報がない場合でも、一般的で有益な旅行アドバイスを提供する
- Contoso Travel がフライト、ホテル、レンタカーの手配をサポートできることを必ず伝える

注意：
あなたはプレミアム旅行代理店である Contoso Travel を代表しています。
回答は常に的確で役立つ内容にしてください。"""

# create_version はエージェントの不変スナップショットを作成する。
# 呼び出すたびに新しいバージョンが生成されます —
# A/B テスト・ロールバック・監査証跡に便利です。
agent = project_client.agents.create_version(
    agent_name="contoso-travel-concierge",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=CONCIERGE_INSTRUCTIONS,
    ),
)

print(f"✅ エージェントを作成しました！")
print(f"   名前: {agent.name}")
print(f"   バージョン: {agent.version}")
print(f"   ID: {agent.id}")

## 3. 会話の開始

会話は複数のメッセージにわたってコンテキストを維持します。会話を作成して最初の旅行クエリを送ってみましょう。

---


In [ ]:
# 会話を作成 — マルチターンコンテキストのために
# メッセージ履歴を追跡するサーバーサイドセッション
conversation = openai_client.conversations.create()
print(f"✅ 会話を作成しました (ID: {conversation.id})")

# extra_body の agent_reference はこのレスポンスを
# 特定の名前付きエージェント（最新バージョンを使用）にバインドする
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="こんにちは！パリ旅行を計画しようと思っているのですが、どんなことを知っておくべきですか？",
)

# output_text: response.output アイテムからすべてのテキストコンテンツを
# 結合する便利なアクセサー
print(f"\n🤖 コンシェルジュの応答:\n")
print(response.output_text)

## 4. マルチターン会話

エージェントは会話内でコンテキストを維持します。フォローアップの質問をして、マルチターン対話の処理方法を確認しましょう。

---


In [ ]:
# 同じ conversation.id = エージェントはコンテキスト全体を保持。
# パリについて再度言及しなくても質問の意図が伝わります。
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="訪れるのに最適な時期はいつですか？",
)

print(f"🤖 コンシェルジュの応答:\n")
print(response.output_text)

In [ ]:
# 第3ターン — エージェントはサーバーサイドのメッセージストアから
# パリに関する会話履歴全体を保持しています
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="1週間の旅行にはどのくらいの予算を見ておくべきか、提案してもらえますか？",
)

print(f"🤖 コンシェルジュの応答:\n")
print(response.output_text)

## 5. レスポンスオブジェクトの確認

レスポンスの構造を見て、どのようなデータが利用可能かを理解しましょう。

---


In [ ]:
# レスポンスオブジェクトの構造を検査。
# output は型付きアイテムのリスト（例: テキストなどの
# コンテンツブロックを含む 'message' アイテム）。
print(f"レスポンス ID:     {response.id}")
print(f"モデル:           {response.model}")
print(f"ステータス:        {response.status}")
print(f"出力テキスト:     {response.output_text[:100]}...")
print(f"\n出力アイテム ({len(response.output)} 件):")
for i, item in enumerate(response.output):
    print(f"  [{i}] タイプ: {item.type}")
    # content は一部のアイテムタイプ（例: reasoning）では None になる
    content = getattr(item, 'content', None)
    if content:
        for j, c in enumerate(content):
            print(f"       Content[{j}]: {type(c).__name__}")

# トークン使用量 — コスト監視のために追跡
print(f"\n使用量:")
print(f"  入力トークン:  {response.usage.input_tokens}")
print(f"  出力トークン: {response.usage.output_tokens}")
print(f"  合計トークン:  {response.usage.total_tokens}")


## 6. クリーンアップ

完了したら常にリソースをクリーンアップしてください — 会話とエージェントバージョンを削除します。

---


In [ ]:
# クリーンアップ: サーバーサイドリソースを解放。
# 会話は明示的に削除するまで残り続けます。
openai_client.conversations.delete(conversation_id=conversation.id)
print("✅ 会話を削除しました")

# このバージョンのみ削除します。同じエージェント名の
# 他のバージョン（存在する場合）は引き続き利用可能です。
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ エージェントバージョンを削除しました")

# HTTP 接続とトークンキャッシュを解放
openai_client.close()
project_client.close()
credential.close()
print("✅ クライアントを閉じました")